# Equilibration Tutorial

This notebook covers the restrained NVT to NPT equilibration protocol. The purpose is to relax bad contacts, thermalize the system near 310 K, and then let the periodic box adapt until the density becomes physically reasonable.

## Thermodynamic Targets

Two ensemble-level checks are central in this stage:

$$eft|angle T 
angle - 310athrm{K}
ight| < 5athrm{K}$$

and

$$0.95 < 
ho < 1.05athrm{gcm^{-3}}$$

These criteria are simple but valuable: they detect thermostats or barostats that are numerically stable yet physically drifting.

In [ ]:
# Resolve project root and import equilibration modules.
from pathlib import Path
import sys
import numpy as np
import openmm
from openmm import XmlSerializer, unit
from openmm.app import PDBFile, Simulation

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_DIR, EquilibrationConfig, MinimizationConfig
from src.physics.restraints import create_positional_restraints
from src.simulate.minimizer import minimize_energy
from src.simulate.equilibrate import run_nvt, run_npt

In [ ]:
# Load the solvated topology and serialized system, then build a Langevin simulation.
topology_pdb = DATA_DIR / 'pdb' / 'prepared' / 'example_solvated.pdb'
system_xml = DATA_DIR / 'topologies' / 'example_system.xml'

pdb = PDBFile(str(topology_pdb))
system = XmlSerializer.deserialize(system_xml.read_text(encoding='utf-8'))
config = EquilibrationConfig(nvt_duration_ps=50.0, npt_duration_ps=100.0, save_interval_ps=2.0)
integrator = openmm.LangevinMiddleIntegrator(
    config.temperature_k * unit.kelvin,
    config.friction_per_ps / unit.picosecond,
    config.timestep_ps * unit.picoseconds,
)
simulation = Simulation(pdb.topology, system, integrator, openmm.Platform.getPlatformByName('CPU'))
simulation.context.setPositions(pdb.positions)
simulation

In [ ]:
# Apply positional restraints to non-hydrogen solute atoms, then minimize.
solute_residue_names = {'HOH', 'WAT', 'NA', 'CL'}
restrained_indices = []
reference_positions = []
for atom_index, atom in enumerate(pdb.topology.atoms()):
    if atom.residue.name.upper() in solute_residue_names:
        continue
    if atom.element is None or atom.element.symbol == 'H':
        continue
    position_nm = pdb.positions[atom_index].value_in_unit(unit.nanometer)
    restrained_indices.append(atom_index)
    reference_positions.append([float(position_nm[0]), float(position_nm[1]), float(position_nm[2])])

create_positional_restraints(
    system,
    restrained_indices,
    np.asarray(reference_positions, dtype=float),
    config.restraint_k_kj_mol_nm2,
)
minimize_energy(simulation, MinimizationConfig(max_iterations=500, tolerance_kj_mol_nm=10.0))

In [ ]:
# Run NVT followed by NPT equilibration and report summary statistics.
equilibration_dir = DATA_DIR / 'trajectories' / 'equilibration'
nvt_result = run_nvt(simulation, config, equilibration_dir / 'nvt_demo')
npt_result = run_npt(simulation, config, equilibration_dir / 'npt_demo')
{
    'nvt_avg_temperature_k': nvt_result['avg_temperature_k'],
    'npt_avg_temperature_k': npt_result['avg_temperature_k'],
    'npt_avg_density_g_cm3': npt_result['avg_density_g_cm3'],
    'npt_final_state_path': npt_result['final_state_path'],
}

## Practical Notes

In production work, equilibration is rarely judged from one scalar alone. Inspect the temperature trace, density relaxation, and any slow box-vector transients. If the system reaches the target density only after a long transient, use the stabilized segment for averaging rather than the full trace.